# Evaluation Architecture Notes

This notebook is a text-style walkthrough of the evaluation layer.

The main idea is:
- the universe stores the shared dataset
- each construction stores fixed weights and in-sample metrics
- backtesting and Monte Carlo are done later
- evaluation results are attached back into the stored construction
- no rolling rebalance and no repeated re-optimization are used here

from portafolios import Universe, local_loader, EqualWeightConstructor, Markowitz, Backtester, MonteCarloEngine

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "portafolios").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "data_clean_stock_data.csv"
YF_SNAPSHOT_PATH = PROJECT_ROOT / "data" / "yf_snapshot.csv"

print("Project root:", PROJECT_ROOT)
print("Processed CSV exists:", CSV_PATH.exists())
print("Yahoo snapshot exists:", YF_SNAPSHOT_PATH.exists())


In [1]:
from portafolios import Universe, local_loader, EqualWeightConstructor, NaiveRiskParity, Markowitz, Backtester, MonteCarloEngine

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "portafolios").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "data_clean_stock_data.csv"
YF_SNAPSHOT_PATH = PROJECT_ROOT / "data" / "yf_snapshot.csv"

print("Project root:", PROJECT_ROOT)
print("Processed CSV exists:", CSV_PATH.exists())
print("Yahoo snapshot exists:", YF_SNAPSHOT_PATH.exists())


Project root: C:\Users\narro\Desktop\semestre\honores_actuaria
Processed CSV exists: True
Yahoo snapshot exists: True


## 2. Build one shared universe

The universe contains:
- prices
- returns
- tickers
- metadata

Here we load a full year of data, but later we will say that the construction window only used the first half of the year.

In [2]:
u = Universe(
    tickers=["AAPL", "MSFT", "AMZN"],
    start="2020-01-01",
    end="2020-12-31",
    loader=local_loader,
    loader_kwargs={"path": CSV_PATH, "prefer_adj_close": True},
    freq="B",
).preparar_datos()

u.prices.head()


,AAPL,MSFT,AMZN
Date,,,
2020-01-02,72.776606,153.428246,94.900497
2020-01-03,72.771729,152.683705,94.309998
2020-01-06,72.621654,151.872308,95.184502
2020-01-07,72.849231,152.416407,95.694504
2020-01-08,73.706271,153.495104,95.550003


In [3]:
print("Universe metadata:", u.metadata)
print("Prices shape:", u.prices.shape)
print("Returns shape:", u.returns.shape)


Universe metadata: {'source': 'callable_dataframe', 'frequency': 'B', 'universe_name': 'universe', 'output_dir': 'outputs\\runs\\universe'}
Prices shape: (261, 3)
Returns shape: (260, 3)


## 3. Build and store constructions

Each construction is saved inside the same universe.

Important detail:
- `construction_start` and `construction_end` identify the estimation window
- this is what lets the evaluation layer validate that the backtest begins later

In [4]:
u.construir(

    NaiveRiskParity(),

    label="equal_weight",

    ann_factor=252,

    construction_start="2020-01-01",

    construction_end="2020-06-30",

)

u.construir(

    NaiveRiskParity(),

    label="naive",

    ann_factor=252,

    construction_start="2020-01-01",

    construction_end="2020-06-30",

)

u.list_constructions()


['equal_weight', 'naive']

In [5]:
stored = u.get_construction("equal_weight")
print("Construction name:", stored.name)
print("Method:", stored.method)
print("Construction window:", stored.construction_start, "->", stored.construction_end)
print("In-sample metrics:", stored.metrics_insample)
stored.weights


Construction name: equal_weight
Method: naive_risk_parity
Construction window: 2020-01-01 00:00:00 -> 2020-06-30 00:00:00
In-sample metrics: {'n_selected': 3, 'expected_return': 0.6095820131793135, 'volatility': 0.3061636977742975, 'sharpe_m': 1.9910329592004554, 'meta_nrp_method': 'inverse_vol', 'meta_nrp_min_vol': 1e-08, 'meta_nrp_sigma': {'AAPL': 0.024012558289591562, 'MSFT': 0.021470874719948713, 'AMZN': 0.021498123067016144}}


AAPL    0.309087
AMZN    0.345238
MSFT    0.345676
Name: weights, dtype: float64

## 4. Fixed-weight backtesting

The backtester:
- retrieves the saved construction
- takes its fixed weights
- checks that the backtest starts after the construction window
- computes realized portfolio returns over the chosen period
- computes cumulative returns and summary metrics
- attaches the result back into the saved construction

In [6]:
bt = Backtester(u, "equal_weight", ann_factor=252)
bt_result = bt.run_and_attach(
    start_date="2020-07-01",
    end_date="2020-12-31",
    notes="Fixed-weight evaluation after the construction window.",
)

bt_result.summary_metrics


{'n_periods': 132,
 'total_return': 0.24314546821709282,
 'annualized_return': 0.5151338381875716,
 'annualized_volatility': 0.27461696267678865,
 'sharpe_ratio': 1.8758267266755118,
 'max_drawdown': -0.1643169821298699}

In [7]:
u.get_construction("equal_weight").backtest_result.summary_metrics


{'n_periods': 132,
 'total_return': 0.24314546821709282,
 'annualized_return': 0.5151338381875716,
 'annualized_volatility': 0.27461696267678865,
 'sharpe_ratio': 1.8758267266755118,
 'max_drawdown': -0.1643169821298699}

## 5. Monte Carlo simulation

The Monte Carlo engine:
- retrieves the same fixed weights
- estimates mean vector and covariance from the universe returns
- simulates future asset returns with a multivariate normal model
- converts them into portfolio paths
- stores the result back inside the same construction

In [8]:
mc = MonteCarloEngine(u, "equal_weight", seed=42)
mc_result = mc.run_and_attach(
    horizon=20,
    n_simulations=500,
    initial_value=1.0,
    notes="Simple multivariate normal Monte Carlo.",
)

mc_result.summary_metrics


{'mean_terminal_value': 1.0364377449437605,
 'median_terminal_value': 1.035570858823351,
 'std_terminal_value': 0.08544392038539546,
 'min_terminal_value': 0.7926198814621547,
 'max_terminal_value': 1.305481235658027,
 'prob_loss': 0.35,
 'mean_terminal_return': 0.036437744943760494}

In [9]:
u.get_construction("equal_weight").mc_result.summary_metrics


{'mean_terminal_value': 1.0364377449437605,
 'median_terminal_value': 1.035570858823351,
 'std_terminal_value': 0.08544392038539546,
 'min_terminal_value': 0.7926198814621547,
 'max_terminal_value': 1.305481235658027,
 'prob_loss': 0.35,
 'mean_terminal_return': 0.036437744943760494}

## 6. Evaluate all stored constructions

The same evaluation layer can run across all saved constructions in the universe.

In [10]:
all_bt = Backtester.run_all(
    u,
    start_date="2020-07-01",
    end_date="2020-12-31",
    ann_factor=252,
    attach=True,
)

{name: result.summary_metrics for name, result in all_bt.items()}


{'equal_weight': {'n_periods': 132,
  'total_return': 0.24314546821709282,
  'annualized_return': 0.5151338381875716,
  'annualized_volatility': 0.27461696267678865,
  'sharpe_ratio': 1.8758267266755118,
  'max_drawdown': -0.1643169821298699},
 'naive': {'n_periods': 132,
  'total_return': 0.24314546821709282,
  'annualized_return': 0.5151338381875716,
  'annualized_volatility': 0.27461696267678865,
  'sharpe_ratio': 1.8758267266755118,
  'max_drawdown': -0.1643169821298699}}

In [11]:
all_mc = MonteCarloEngine.run_all(
    u,
    horizon=20,
    n_simulations=250,
    initial_value=1.0,
    attach=True,
    seed=123,
)

{name: result.summary_metrics for name, result in all_mc.items()}


{'equal_weight': {'mean_terminal_value': 1.0320101865685674,
  'median_terminal_value': 1.034917094712371,
  'std_terminal_value': 0.08514188620950697,
  'min_terminal_value': 0.8173904395246071,
  'max_terminal_value': 1.2379381438187624,
  'prob_loss': 0.388,
  'mean_terminal_return': 0.032010186568567474},
 'naive': {'mean_terminal_value': 1.03404483058898,
  'median_terminal_value': 1.0333984209076448,
  'std_terminal_value': 0.07730836369176611,
  'min_terminal_value': 0.8171119884869957,
  'max_terminal_value': 1.2549182376080168,
  'prob_loss': 0.316,
  'mean_terminal_return': 0.03404483058898011}}

## 7. Final interpretation

The architecture is now separated in three layers:

1. Data layer
   - loads and standardizes prices and returns

2. Construction layer
   - creates fixed weights and in-sample metrics
   - stores each construction in the universe

3. Evaluation layer
   - backtests fixed weights after the construction window
   - runs Monte Carlo on the saved fixed weights
   - attaches both results back to the corresponding construction